# Dataset Refiner

Turns the raw TMDB scrape into the modelling dataset.

**Input:** `reference_docs/box_office_2021_plus.csv` (written by `scraper.py`, ~23k US theatrical releases)
**Output:** `reference_docs/box_office_refined.csv`, which goes into `starpower.py` next

**Steps**
1. **Filter** to films with usable financials and metadata.
2. **Build reference tables** from TMDB for each remaining film: every US release date (with its note), plus original language, origin country, keywords and current US streaming providers. They're cached to CSV in `reference_docs/reference_tables/`, and reruns only fetch films that aren't in them yet.
3. **Flag Netflix releases** where two deterministic sources agree: the TMDB release-date table and Wikipedia's curated Netflix film lists (scraped into a third reference table).
4. **Flag anime** (also adding `Anime` to the `Genres` string) and **flag Chinese-language films**. These are flags, not filters: `ml_explore.ipynb` has a toggle for each one.
5. **Review** the cases where the rules and secondary sources disagree, and fix them with the override CSVs.
6. **Write** the refined CSV.

Every rule is a plain function with its thresholds in the **Config** cell. The override CSVs (`netflix_overrides.csv`, `anime_overrides.csv`, `chinese_overrides.csv`) always have the final word, the same way `star_overrides.csv` does for star power.

## 1. Imports & config

In [ ]:
import os
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import date
from io import StringIO
from pathlib import Path
from urllib.parse import unquote

import pandas as pd
import requests
from dotenv import load_dotenv
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from scraper import TMDB  # same cached, rate-limit-aware client the scraper uses

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

In [ ]:
# ---- Files ----
RAW_SCRAPE_PATH = Path("reference_docs/box_office_2021_plus.csv")
OUTPUT_PATH = Path("reference_docs/box_office_refined.csv")

REFERENCE_DIR = Path("reference_docs/reference_tables")
TMDB_FACTS_PATH = REFERENCE_DIR / "tmdb_movie_facts.csv"            # one row per film
TMDB_RELEASES_PATH = REFERENCE_DIR / "tmdb_us_release_dates.csv"    # one row per US release event
WIKIPEDIA_NETFLIX_PATH = REFERENCE_DIR / "wikipedia_netflix_films.csv"  # one row per Wikipedia list entry

NETFLIX_OVERRIDES_PATH = Path("netflix_overrides.csv")
ANIME_OVERRIDES_PATH = Path("anime_overrides.csv")
CHINESE_OVERRIDES_PATH = Path("chinese_overrides.csv")

# ---- Filtering ----
MIN_BUDGET = 1_000_000   # budget must be strictly greater than this (TMDB has many placeholder budgets like $5)
REQUIRED_COLUMNS = ["Worldwide", "Rating", "Genres", "Production Companies"]

# ---- TMDB fetching ----
TMDB_CACHE_DIR = Path(".cache/tmdb")
TMDB_CACHE_HOURS = 24 * 7         # raw API responses older than this are re-downloaded
TMDB_WORKERS = 8
REFRESH_REFERENCE_TABLES = False  # True = rebuild the TMDB tables from scratch; False = only fetch new films

# ---- Netflix rule (both conditions must hold, see section 5) ----
NETFLIX_MAX_WINDOW_DAYS = 60      # TMDB: US Netflix digital release within this many days of theatrical
NETFLIX_LIST_MIN_YEAR = 2020      # Wikipedia: scrape "List of Netflix original films (...)" pages from this year on
NETFLIX_EXTRA_PAGES = ["List of Netflix exclusive international distribution films"]
REFRESH_WIKIPEDIA = False         # True = re-scrape Wikipedia; False = reuse WIKIPEDIA_NETFLIX_PATH

WIKIPEDIA_API = "https://en.wikipedia.org/w/api.php"
WIKIDATA_API = "https://www.wikidata.org/w/api.php"
WIKI_USER_AGENT = "box-office-tools/1.0 (personal research script)"
WIKI_PAUSE_SECONDS = 1.0          # Wikipedia rate-limits hard; keep at least ~1s between calls

# ---- Anime rule ----
# A film is anime if TMDB tags it Animation AND (it has the "anime" keyword OR it is Japanese by language/origin).
ANIME_KEYWORDS = {"anime"}
ANIME_LANGUAGES = {"ja"}
ANIME_COUNTRIES = {"JP"}
ANIME_GENRE_NAME = "Anime"

# ---- Chinese-film rule ----
# A film is Chinese if its original language is Mandarin/Cantonese OR its origin country is China / Hong Kong / Taiwan.
CHINESE_LANGUAGES = {"zh", "cn"}     # TMDB uses "cn" for Cantonese
CHINESE_COUNTRIES = {"CN", "HK", "TW"}

# TMDB release types
RELEASE_TYPE_NAMES = {1: "premiere", 2: "theatrical_limited", 3: "theatrical", 4: "digital", 5: "physical", 6: "tv"}
THEATRICAL_TYPES = {2, 3}
DIGITAL_TYPE = 4

## 2. Load the raw scrape

In [ ]:
def load_raw_scrape(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["Release Date"] = pd.to_datetime(df["Release Date"])
    return df

In [ ]:
raw = load_raw_scrape(RAW_SCRAPE_PATH)
print(f"{len(raw):,} films in the raw scrape")
print("\nNon-missing values per column:")
print(raw.notna().sum())

## 3. Filter to modelable films

This reproduces the filter behind the original `box_office_filtered.csv` (931 films):
- `Budget` > `MIN_BUDGET`
- `Worldwide`, `Rating`, `Genres` and `Production Companies` are all present

Each condition is applied one at a time, and the step-by-step counts show what each one removes.

In [ ]:
def filter_modelable_films(df: pd.DataFrame, min_budget: float, required_columns: list[str]) -> pd.DataFrame:
    print(f"{'start':<32}{len(df):>7,}")

    df = df[df["Budget"] > min_budget]
    print(f"{f'Budget > ${min_budget:,.0f}':<32}{len(df):>7,}")

    for col in required_columns:
        df = df[df[col].notna()]
        print(f"{f'{col} present':<32}{len(df):>7,}")

    return df.sort_values(["Release Date", "Movie"]).reset_index(drop=True)

In [ ]:
films = filter_modelable_films(raw, MIN_BUDGET, REQUIRED_COLUMNS)
films.head()

## 4. TMDB reference tables

For each film, one `/movie/{id}` call (with `release_dates`, `keywords` and `watch/providers` appended) fills two tables:

- **`tmdb_movie_facts.csv`**: one row per film with `original_language`, `origin_country`, `keywords` and `us_flatrate_providers` (where it streams in the US *today*). Multi-value fields are `|`-separated.
- **`tmdb_us_release_dates.csv`**: one row per US release event with its `type` (premiere / limited / wide theatrical / digital / physical / TV), `date` and free-text `note`. Netflix releases show up here as an early digital release with the note "Netflix".

Both tables are saved, so the rules below run on a fixed snapshot. Reruns only fetch films that aren't in the tables yet. Set `REFRESH_REFERENCE_TABLES = True` to rebuild them.

Needs `TMDB_TOKEN` or `TMDB_API_KEY` in `.env`, the same as `scraper.py`.

In [ ]:
def make_tmdb_client() -> TMDB:
    load_dotenv(Path(".env"))
    token = os.getenv("TMDB_TOKEN", "").strip() or None
    api_key = os.getenv("TMDB_API_KEY", "").strip() or None
    if not token and not api_key:
        raise RuntimeError("Set TMDB_TOKEN or TMDB_API_KEY in .env")
    return TMDB(token, api_key, TMDB_CACHE_DIR, TMDB_CACHE_HOURS)


def fetch_tmdb_movie(tmdb: TMDB, tmdb_id: int) -> dict | None:
    return tmdb.get(f"/movie/{tmdb_id}", append_to_response="release_dates,keywords,watch/providers")

In [ ]:
def parse_movie_facts(movie: dict) -> dict:
    us_providers = movie.get("watch/providers", {}).get("results", {}).get("US", {})
    return {
        "tmdb_id": movie["id"],
        "title": movie.get("title"),
        "original_language": movie.get("original_language") or "",
        "origin_country": "|".join(movie.get("origin_country", [])),
        "keywords": "|".join(k["name"] for k in movie.get("keywords", {}).get("keywords", [])),
        "us_flatrate_providers": "|".join(p["provider_name"] for p in us_providers.get("flatrate", [])),
        "fetched": date.today().isoformat(),
    }


def parse_us_release_dates(movie: dict) -> list[dict]:
    countries = movie.get("release_dates", {}).get("results", [])
    us = next((c for c in countries if c["iso_3166_1"] == "US"), None)
    if us is None:
        return []
    return [
        {
            "tmdb_id": movie["id"],
            "type": r["type"],
            "type_name": RELEASE_TYPE_NAMES.get(r["type"], "unknown"),
            "date": r["release_date"][:10],
            "note": (r.get("note") or "").strip(),
            "certification": r.get("certification") or "",
        }
        for r in us["release_dates"]
    ]

In [ ]:
FACTS_COLUMNS = ["tmdb_id", "title", "original_language", "origin_country", "keywords", "us_flatrate_providers", "fetched"]
RELEASES_COLUMNS = ["tmdb_id", "type", "type_name", "date", "note", "certification"]


def read_table(path: Path, columns: list[str]) -> pd.DataFrame:
    """Read a saved reference table. Text fields stay as "" instead of NaN."""
    if not path.exists():
        return pd.DataFrame(columns=columns)
    return pd.read_csv(path, dtype=str, keep_default_na=False).astype({"tmdb_id": int})


def update_tmdb_reference_tables(tmdb: TMDB, tmdb_ids: list[int], refresh: bool) -> tuple[pd.DataFrame, pd.DataFrame]:
    facts = pd.DataFrame(columns=FACTS_COLUMNS) if refresh else read_table(TMDB_FACTS_PATH, FACTS_COLUMNS)
    releases = pd.DataFrame(columns=RELEASES_COLUMNS) if refresh else read_table(TMDB_RELEASES_PATH, RELEASES_COLUMNS)

    missing = sorted(set(tmdb_ids) - set(facts["tmdb_id"]))
    print(f"{len(tmdb_ids) - len(missing)} films already in the reference tables, fetching {len(missing)}")

    new_facts, new_releases, failures = [], [], []
    with ThreadPoolExecutor(max_workers=TMDB_WORKERS) as pool:
        futures = {pool.submit(fetch_tmdb_movie, tmdb, tmdb_id): tmdb_id for tmdb_id in missing}
        for i, future in enumerate(as_completed(futures), start=1):
            tmdb_id = futures[future]
            try:
                movie = future.result()
            except Exception as exc:  # keep going; report at the end
                failures.append((tmdb_id, repr(exc)))
                continue
            if movie is None:
                failures.append((tmdb_id, "404 not found on TMDB"))
                continue
            new_facts.append(parse_movie_facts(movie))
            new_releases.extend(parse_us_release_dates(movie))
            if i % 200 == 0 or i == len(futures):
                print(f"  {i}/{len(futures)}")

    if new_facts:
        facts = pd.concat([facts, pd.DataFrame(new_facts, columns=FACTS_COLUMNS)], ignore_index=True)
        releases = pd.concat([releases, pd.DataFrame(new_releases, columns=RELEASES_COLUMNS)], ignore_index=True)
        facts = facts.sort_values("tmdb_id").reset_index(drop=True)
        releases = releases.sort_values(["tmdb_id", "date", "type"]).reset_index(drop=True)

    REFERENCE_DIR.mkdir(parents=True, exist_ok=True)
    facts.to_csv(TMDB_FACTS_PATH, index=False)
    releases.to_csv(TMDB_RELEASES_PATH, index=False)

    if failures:
        print(f"{len(failures)} films failed (rerun to retry): {failures[:10]}")
    return facts, releases

In [ ]:
tmdb = make_tmdb_client()
facts, releases = update_tmdb_reference_tables(tmdb, films["TMDB ID"].tolist(), REFRESH_REFERENCE_TABLES)

# Only keep rows for films in the current filtered set (the tables may hold extra films from earlier runs)
facts = facts[facts["tmdb_id"].isin(films["TMDB ID"])].reset_index(drop=True)
releases = releases[releases["tmdb_id"].isin(films["TMDB ID"])].reset_index(drop=True)
releases["type"] = releases["type"].astype(int)
releases["date"] = pd.to_datetime(releases["date"])

print(f"\nfacts: {len(facts)} films   releases: {len(releases)} US release events")

In [ ]:
# Peek at one film's US release history
releases[releases["tmdb_id"] == 1062722]  # Frankenstein (2025): limited theatrical, then Netflix three weeks later

## 5. Netflix releases

A film counts as a Netflix release only when **two independent sources agree**:

1. **TMDB release window:** a US digital release (type 4) whose note mentions *Netflix*, no more than `NETFLIX_MAX_WINDOW_DAYS` after the first US theatrical date. This separates Netflix's own films (*Frankenstein*: limited release, then Netflix 21 days later) from studio films licensed long after their run (*Demon Slayer: Infinity Castle* reached Netflix about a year later).
2. **Wikipedia's curated Netflix lists:** the film appears on a "List of Netflix original films" page, or on "List of Netflix exclusive international distribution films" with a region that includes the US.

Why both? Each one alone has false positives:
- TMDB alone also catches Indian theatrical releases (*RRR*, *Animal*, *Dhurandhar*). Their standard 4–8 week theatrical window often ends on Netflix.
- Wikipedia alone catches late acquisitions, e.g. *Black Box* (2022 theatrical) listed as a 2026 Netflix film.

### 5a. TMDB release-window signal

In [ ]:
def digital_releases_with_window(releases: pd.DataFrame) -> pd.DataFrame:
    """US digital releases, each with the number of days after the film's first US theatrical date."""
    first_theatrical = (
        releases[releases["type"].isin(THEATRICAL_TYPES)]
        .groupby("tmdb_id")["date"].min()
        .rename("first_theatrical")
    )
    digital = releases[releases["type"] == DIGITAL_TYPE].join(first_theatrical, on="tmdb_id")
    digital["days_after_theatrical"] = (digital["date"] - digital["first_theatrical"]).dt.days
    return digital


def tmdb_netflix_window_matches(releases: pd.DataFrame, max_window_days: int) -> pd.DataFrame:
    """One row per film with a US Netflix digital release inside the window (the earliest such release)."""
    digital = digital_releases_with_window(releases)
    matches = digital[
        digital["note"].str.contains("netflix", case=False)
        & (digital["days_after_theatrical"] <= max_window_days)
    ]
    return matches.sort_values("date").drop_duplicates("tmdb_id").reset_index(drop=True)

In [ ]:
tmdb_netflix = tmdb_netflix_window_matches(releases, NETFLIX_MAX_WINDOW_DAYS)
tmdb_netflix_ids = set(tmdb_netflix["tmdb_id"])

print(f"{len(tmdb_netflix_ids)} films have a US Netflix digital release within {NETFLIX_MAX_WINDOW_DAYS} days of theatrical")
(
    tmdb_netflix.merge(films[["TMDB ID", "Movie", "Worldwide"]], left_on="tmdb_id", right_on="TMDB ID")
    [["Movie", "first_theatrical", "date", "days_after_theatrical", "note", "Worldwide"]]
    .rename(columns={"date": "digital_date"})
)

### 5b. Wikipedia Netflix lists (reference table)

This scrapes the Wikipedia list pages and resolves each linked film article to a **TMDB ID** through Wikidata (article → Wikidata item → property P4947, "TMDB movie ID"). The match is by ID, not by title.

- The year pages ("List of Netflix original films (2025)", "(since 2026)", …) are discovered automatically, so new years get picked up.
- On the international-distribution page, the "Netflix exclusive region" column decides whether the US is covered (`region_includes_us`).
- Titles without a Wikipedia article can't be resolved. They stay in the table with no `tmdb_id`. Use `netflix_overrides.csv` if one of them matters.

Wikipedia rate-limits aggressively, so `WikiClient` pauses between calls and retries on HTTP 429. The result is saved to `wikipedia_netflix_films.csv` and reused until `REFRESH_WIKIPEDIA = True`.

In [ ]:
class WikiClient:
    """Minimal MediaWiki API client: polite pacing plus retries that honour Retry-After."""

    def __init__(self, pause_seconds: float):
        self.pause_seconds = pause_seconds
        self.session = requests.Session()
        self.session.headers["User-Agent"] = WIKI_USER_AGENT
        retry = Retry(total=8, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504],
                      allowed_methods=None, respect_retry_after_header=True)
        self.session.mount("https://", HTTPAdapter(max_retries=retry))

    def call(self, api_url: str, **params) -> dict:
        time.sleep(self.pause_seconds)
        # POST so that long batched title lists don't hit URL-length limits
        response = self.session.post(api_url, data={**params, "format": "json", "formatversion": 2}, timeout=60)
        response.raise_for_status()
        return response.json()

In [ ]:
def discover_netflix_list_pages(wiki: WikiClient, min_year: int) -> list[str]:
    """Year pages of "List of Netflix original films (...)" covering min_year or later, plus the extra pages."""
    data = wiki.call(WIKIPEDIA_API, action="query", list="allpages", apprefix="List of Netflix original films (",
                     apfilterredir="nonredirects", aplimit=100)
    pages = []
    for title in (p["title"] for p in data["query"]["allpages"]):
        years = [int(y) for y in re.findall(r"\d{4}", title)]
        if years and max(years) >= min_year:
            pages.append(title)
    return sorted(pages) + NETFLIX_EXTRA_PAGES


def article_from_href(href: str | None) -> str | None:
    """'/wiki/Frankenstein_(2025_film)' -> 'Frankenstein (2025 film)'. Red links and external links -> None."""
    if not isinstance(href, str) or not href.startswith("/wiki/"):
        return None
    return unquote(href[len("/wiki/"):].split("#")[0]).replace("_", " ")


def scrape_list_page(wiki: WikiClient, page: str) -> pd.DataFrame:
    html = wiki.call(WIKIPEDIA_API, action="parse", page=page, prop="text", redirects=1)["parse"]["text"]
    rows = []
    for table in pd.read_html(StringIO(html), extract_links="body"):
        if "Title" not in table.columns:
            continue
        regions = table["Netflix exclusive region"] if "Netflix exclusive region" in table.columns else None
        for i, (title, href) in enumerate(table["Title"]):
            rows.append({
                "page": page,
                "title": title,
                "article": article_from_href(href),
                "netflix_region": regions.iloc[i][0] if regions is not None else "",
            })
    return pd.DataFrame(rows)

In [ ]:
def articles_to_wikidata_ids(wiki: WikiClient, articles: list[str]) -> dict[str, str]:
    """Map Wikipedia article titles to Wikidata item IDs (Q...), following redirects. 50 titles per call."""
    mapping = {}
    for i in range(0, len(articles), 50):
        batch = articles[i:i + 50]
        query = wiki.call(WIKIPEDIA_API, action="query", titles="|".join(batch), prop="pageprops",
                          ppprop="wikibase_item", redirects=1)["query"]
        normalized = {n["from"]: n["to"] for n in query.get("normalized", [])}
        redirects = {r["from"]: r["to"] for r in query.get("redirects", [])}
        item_by_title = {p["title"]: p.get("pageprops", {}).get("wikibase_item") for p in query["pages"]}
        for article in batch:
            resolved = normalized.get(article, article)
            resolved = redirects.get(resolved, resolved)
            if item_by_title.get(resolved):
                mapping[article] = item_by_title[resolved]
    return mapping


def wikidata_ids_to_tmdb_ids(wiki: WikiClient, qids: list[str]) -> dict[str, int]:
    """Map Wikidata item IDs to TMDB movie IDs (property P4947). 50 items per call."""
    mapping = {}
    for i in range(0, len(qids), 50):
        entities = wiki.call(WIKIDATA_API, action="wbgetentities", ids="|".join(qids[i:i + 50]), props="claims")["entities"]
        for qid, entity in entities.items():
            claims = entity.get("claims", {}).get("P4947", [])
            if claims:
                mapping[qid] = int(claims[0]["mainsnak"]["datavalue"]["value"])
    return mapping


def region_includes_us(region: str) -> bool:
    """Original-film pages have no region (Netflix holds every territory). Otherwise parse the region text."""
    region = region.lower().strip()
    if not region:
        return True
    us_terms = ("united states", "north america")
    if region.startswith("worldwide"):
        excluded = region.split("except", 1)[1] if "except" in region else ""
        return not any(term in excluded for term in us_terms)
    return any(term in region for term in us_terms)

In [ ]:
def build_wikipedia_netflix_table(wiki: WikiClient, min_year: int) -> pd.DataFrame:
    pages = discover_netflix_list_pages(wiki, min_year)
    print(f"Scraping {len(pages)} pages: {pages}")
    table = pd.concat([scrape_list_page(wiki, page) for page in pages], ignore_index=True)

    articles = sorted(table["article"].dropna().unique())
    print(f"{len(table)} rows, {len(articles)} linked articles. Resolving to TMDB IDs...")
    article_to_qid = articles_to_wikidata_ids(wiki, articles)
    qid_to_tmdb = wikidata_ids_to_tmdb_ids(wiki, sorted(set(article_to_qid.values())))

    table["wikidata_id"] = table["article"].map(article_to_qid)
    table["tmdb_id"] = table["wikidata_id"].map(qid_to_tmdb).astype("Int64")
    table["us_netflix"] = table["netflix_region"].map(region_includes_us)
    table["fetched"] = date.today().isoformat()
    return table


def load_wikipedia_netflix(path: Path, min_year: int, refresh: bool) -> pd.DataFrame:
    if path.exists() and not refresh:
        table = pd.read_csv(path, keep_default_na=False, dtype=str)
        table["tmdb_id"] = pd.to_numeric(table["tmdb_id"]).astype("Int64")
        table["us_netflix"] = table["us_netflix"].eq("True")
        print(f"Using Wikipedia snapshot {path.name} (fetched {table['fetched'].iloc[0]}, {len(table)} rows)")
        return table
    table = build_wikipedia_netflix_table(WikiClient(WIKI_PAUSE_SECONDS), min_year)
    REFERENCE_DIR.mkdir(parents=True, exist_ok=True)
    table.to_csv(path, index=False)
    return table

In [ ]:
wikipedia_netflix = load_wikipedia_netflix(WIKIPEDIA_NETFLIX_PATH, NETFLIX_LIST_MIN_YEAR, REFRESH_WIKIPEDIA)

resolved = wikipedia_netflix.dropna(subset=["tmdb_id"])
wikipedia_any_ids = set(resolved["tmdb_id"].astype(int)) & set(films["TMDB ID"])
wikipedia_us_ids = set(resolved.loc[resolved["us_netflix"], "tmdb_id"].astype(int)) & set(films["TMDB ID"])

print(f"{resolved['tmdb_id'].nunique()} listed films resolved to a TMDB ID "
      f"({wikipedia_netflix['tmdb_id'].isna().sum()} rows unresolved)")
print(f"In our dataset: {len(wikipedia_any_ids)} listed, {len(wikipedia_us_ids)} with Netflix holding the US")

### 5c. Combine the two signals and review

`netflix_rule_ids` = films in **both** the TMDB window set and the Wikipedia US set.

The review table lists every film that any source points at. Rows where TMDB and Wikipedia disagree come first. If a disagreement turns out to be a real miss, add a row to `netflix_overrides.csv`.
- `wikipedia_region`: the region text, empty for Netflix originals
- `streams_on_netflix_now`: Netflix is a current US subscription provider (context only; licensed films show up here too)
- `first_digital_note` / `days_to_first_digital`: the earliest US digital release and how long after theatrical it came

In [ ]:
netflix_rule_ids = tmdb_netflix_ids & wikipedia_us_ids
print(f"{len(netflix_rule_ids)} films flagged as Netflix releases (both sources agree)")

In [ ]:
def netflix_review_table(films, facts, releases, wikipedia_netflix, tmdb_ids, wikipedia_us_ids, wikipedia_any_ids) -> pd.DataFrame:
    streams_now = set(facts.loc[facts["us_flatrate_providers"].str.contains("Netflix"), "tmdb_id"])
    language = facts.set_index("tmdb_id")["original_language"]

    first_digital = (
        digital_releases_with_window(releases)
        .sort_values("date")
        .drop_duplicates("tmdb_id")
        .set_index("tmdb_id")[["note", "days_after_theatrical"]]
        .rename(columns={"note": "first_digital_note", "days_after_theatrical": "days_to_first_digital"})
    )
    region = (
        wikipedia_netflix.dropna(subset=["tmdb_id"])
        .astype({"tmdb_id": int})
        .drop_duplicates("tmdb_id")
        .set_index("tmdb_id")["netflix_region"]
        .rename("wikipedia_region")
    )

    table = films.loc[films["TMDB ID"].isin(tmdb_ids | wikipedia_any_ids), ["TMDB ID", "Movie", "Release Date", "Worldwide"]].copy()
    table["language"] = table["TMDB ID"].map(language)
    table["tmdb_window"] = table["TMDB ID"].isin(tmdb_ids)
    table["wikipedia_us"] = table["TMDB ID"].isin(wikipedia_us_ids)
    table["streams_on_netflix_now"] = table["TMDB ID"].isin(streams_now)
    table = table.join(first_digital, on="TMDB ID").join(region, on="TMDB ID")
    table["netflix"] = table["tmdb_window"] & table["wikipedia_us"]
    table["sources_agree"] = table["tmdb_window"] == table["wikipedia_us"]
    return table.sort_values(["sources_agree", "netflix", "Release Date"], ascending=[True, False, True]).reset_index(drop=True)

In [ ]:
netflix_review = netflix_review_table(films, facts, releases, wikipedia_netflix,
                                      tmdb_netflix_ids, wikipedia_us_ids, wikipedia_any_ids)
print(f"{len(netflix_review)} candidate films, {(~netflix_review['sources_agree']).sum()} where the sources disagree")
netflix_review[~netflix_review["sources_agree"]]

In [ ]:
# The films flagged as Netflix releases
netflix_review[netflix_review["netflix"]]

## 6. Anime

**Rule:** a film is anime if TMDB's genres include **Animation** AND at least one of:
- it has TMDB's `anime` keyword (this also catches non-Japanese-language anime such as *The Lord of the Rings: The War of the Rohirrim*)
- its original language is Japanese
- its origin country is Japan

Requiring Animation keeps out live-action manga adaptations and Japanese live-action films.

In [ ]:
def split_pipe(value: str) -> set[str]:
    return {v.strip() for v in value.split("|") if v.strip()}


def anime_signals(films: pd.DataFrame, facts: pd.DataFrame) -> pd.DataFrame:
    """One row per film with each anime signal as its own True/False column."""
    signals = films[["TMDB ID", "Movie", "Genres"]].merge(facts, left_on="TMDB ID", right_on="tmdb_id", how="left")
    signals = signals.fillna({"keywords": "", "original_language": "", "origin_country": ""})
    signals["animation_genre"] = signals["Genres"].str.split(", ").map(lambda g: "Animation" in g)
    signals["anime_keyword"] = signals["keywords"].str.lower().map(split_pipe).map(lambda k: bool(k & ANIME_KEYWORDS))
    signals["japanese_language"] = signals["original_language"].isin(ANIME_LANGUAGES)
    signals["japanese_origin"] = signals["origin_country"].map(split_pipe).map(lambda c: bool(c & ANIME_COUNTRIES))
    signals["anime_rule"] = signals["animation_genre"] & (
        signals["anime_keyword"] | signals["japanese_language"] | signals["japanese_origin"]
    )
    return signals[["TMDB ID", "Movie", "Genres", "original_language", "origin_country",
                    "animation_genre", "anime_keyword", "japanese_language", "japanese_origin", "anime_rule"]]

In [ ]:
anime = anime_signals(films, facts)
anime_rule_ids = set(anime.loc[anime["anime_rule"], "TMDB ID"])
print(f"{len(anime_rule_ids)} films match the anime rule")
anime[anime["anime_rule"]]

In [ ]:
# Near misses: some anime signal but the rule says no (e.g. the anime keyword without the Animation genre).
# Japanese live-action films land here by design. Add an override if one should count.
anime[~anime["anime_rule"] & (anime["anime_keyword"] | (anime["animation_genre"] & anime["origin_country"].str.contains("JP")))]

## 7. Chinese-language films

**Rule:** original language Mandarin or Cantonese (`zh` / `cn`) OR origin country China, Hong Kong or Taiwan.

These films earn most of their gross in their home market (*Ne Zha 2*: $2.27B worldwide, a small share of it in the US), so a worldwide-gross model fitted on them doesn't transfer well to US-focused predictions. They're flagged here and **excluded in `ml_explore.ipynb` by default** (`EXCLUDE_CHINESE`).

In [ ]:
def chinese_signals(films: pd.DataFrame, facts: pd.DataFrame) -> pd.DataFrame:
    """One row per film with each Chinese-film signal as its own True/False column."""
    signals = films[["TMDB ID", "Movie", "Worldwide"]].merge(facts, left_on="TMDB ID", right_on="tmdb_id", how="left")
    signals = signals.fillna({"original_language": "", "origin_country": ""})
    signals["chinese_language"] = signals["original_language"].isin(CHINESE_LANGUAGES)
    signals["chinese_origin"] = signals["origin_country"].map(split_pipe).map(lambda c: bool(c & CHINESE_COUNTRIES))
    signals["chinese_rule"] = signals["chinese_language"] | signals["chinese_origin"]
    return signals[["TMDB ID", "Movie", "Worldwide", "original_language", "origin_country",
                    "chinese_language", "chinese_origin", "chinese_rule"]]

In [ ]:
chinese = chinese_signals(films, facts)
chinese_rule_ids = set(chinese.loc[chinese["chinese_rule"], "TMDB ID"])
print(f"{len(chinese_rule_ids)} films match the Chinese-film rule")
chinese[chinese["chinese_rule"]]

## 8. Overrides

Overrides are the final word, the same as `star_overrides.csv`. Each file has the columns `tmdb_id, movie, value, note`. `value` is 1 to force the flag on and 0 to force it off. `movie` and `note` are only there for readability.

In [ ]:
def load_flag_overrides(path: Path) -> dict[int, bool]:
    if not path.exists():
        print(f"{path.name}: not found, no overrides")
        return {}
    table = pd.read_csv(path, dtype={"tmdb_id": int, "value": int})
    bad = table[~table["value"].isin([0, 1])]
    if len(bad):
        raise ValueError(f"{path.name}: value must be 0 or 1. Fix: {bad.to_dict('records')}")
    print(f"{path.name}: {len(table)} overrides")
    return dict(zip(table["tmdb_id"], table["value"].astype(bool)))


def apply_flag(films: pd.DataFrame, rule_ids: set[int], overrides: dict[int, bool]) -> tuple[pd.Series, pd.Series]:
    """Return (flag, source) Series aligned to films. Source is 'rule', 'override' or ''."""
    flag = films["TMDB ID"].isin(rule_ids)
    source = flag.map({True: "rule", False: ""})
    overridden = films["TMDB ID"].isin(overrides)
    flag[overridden] = films.loc[overridden, "TMDB ID"].map(overrides)
    source[overridden] = "override"
    return flag, source

In [ ]:
netflix_overrides = load_flag_overrides(NETFLIX_OVERRIDES_PATH)
anime_overrides = load_flag_overrides(ANIME_OVERRIDES_PATH)
chinese_overrides = load_flag_overrides(CHINESE_OVERRIDES_PATH)

## 9. Build the refined dataset

New columns added to the scrape's columns:
| Column | Meaning |
|---|---|
| `Is Netflix` | 1 = Netflix release (rule or override) |
| `Is Anime` | 1 = anime (rule or override). `Anime` is also appended to `Genres` |
| `Is Chinese` | 1 = Mandarin/Cantonese-language or Chinese/HK/Taiwanese-origin film (rule or override) |
| `Original Language` | TMDB original language code (`en`, `ja`, `ko`, …) |
| `US Wide Release` | 1 = TMDB has a wide (type 3) US theatrical release, 0 = limited only |
| `Flag Source` | where each flag came from, e.g. `netflix:rule; anime:override`, for auditing |

In [ ]:
def add_genre(genres: str, genre: str) -> str:
    parts = genres.split(", ")
    return genres if genre in parts else ", ".join(parts + [genre])


def wide_release_ids(releases: pd.DataFrame) -> set[int]:
    return set(releases.loc[releases["type"] == 3, "tmdb_id"])


def build_refined_dataset(films, facts, releases, flags: dict[str, tuple[pd.Series, pd.Series]]) -> pd.DataFrame:
    """`flags` maps a flag name ("netflix", "anime", "chinese") to its (flag, source) pair from apply_flag."""
    refined = films.copy()

    for name, (flag, _) in flags.items():
        refined[f"Is {name.capitalize()}"] = flag.astype(int)

    anime_flag = flags["anime"][0]
    refined.loc[anime_flag, "Genres"] = refined.loc[anime_flag, "Genres"].map(lambda g: add_genre(g, ANIME_GENRE_NAME))

    language = facts.set_index("tmdb_id")["original_language"]
    refined["Original Language"] = refined["TMDB ID"].map(language)
    refined["US Wide Release"] = refined["TMDB ID"].isin(wide_release_ids(releases)).astype(int)

    sources = pd.DataFrame({name: source for name, (_, source) in flags.items()})
    refined["Flag Source"] = sources.apply(
        lambda row: "; ".join(f"{name}:{src}" for name, src in row.items() if src), axis=1
    )
    return refined

In [ ]:
flags = {
    "netflix": apply_flag(films, netflix_rule_ids, netflix_overrides),
    "anime": apply_flag(films, anime_rule_ids, anime_overrides),
    "chinese": apply_flag(films, chinese_rule_ids, chinese_overrides),
}
refined = build_refined_dataset(films, facts, releases, flags)

print(f"{len(refined)} films")
print(f"  Is Netflix:      {refined['Is Netflix'].sum()}")
print(f"  Is Anime:        {refined['Is Anime'].sum()}")
print(f"  Is Chinese:      {refined['Is Chinese'].sum()}")
print(f"  US Wide Release: {refined['US Wide Release'].sum()}")
print(f"  Missing language (not in reference tables): {refined['Original Language'].isna().sum()}")

In [ ]:
# Sanity check: typical gross by group. Netflix films should look very different from everything else.
refined.groupby(["Is Netflix", "US Wide Release"])["Worldwide"].agg(films="count", median_worldwide="median").round(0)

## 10. Write output

In [ ]:
refined.to_csv(OUTPUT_PATH, index=False)
print(f"Wrote {len(refined)} films to {OUTPUT_PATH}")

**Next step:** add star power, then open `ml_explore.ipynb`:

```
python starpower.py reference_docs/box_office_refined.csv
```

This writes `reference_docs/box_office_refined_starpower.csv`, which is the file `ml_explore.ipynb` reads.